# 07 · Targeted fixes for the weakest models
Only the models below 65% accuracy are touched. The other models in `models/final/` stay as they are.

| Model | v2 test | Fix (chosen with train-only validation experiments) |
|---|---|---|
| product `break_even_months` | R² 0.49, 57% | Launch + traction features **without** the noisy track-record encodings; smaller leaves; **capped at 36 months** (only ~6% of products take longer; the app shows them as "36+ months", and the separate `never_breaks_even` model flags the non-payers) |
| product `demand_share_pct_12m` | 54% | **Composite:** predicted avg monthly demand ÷ the category's total monthly demand (market size) during the product's first year |
| yearly `net_profit` | 60% | **Composite:** revenue × predicted margin, where revenue = last year's revenue × predicted growth factor (direct revenue model for first-year products) |
| monthly / yearly demand growth % | 47% / 39% | **Not fixable** (see section 5): the year-to-year and month-to-month swings are mostly the random noise built into the data |

Accuracy = 100 − (total absolute error ÷ total absolute actual value) × 100. The test set is used once, in section 4.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.ticker as mtick, seaborn as sns
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

# One consistent, colour-blind-checked palette for every chart (fixed order: colour follows the entity)
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQ = sns.blend_palette(["#cde2fb", "#2a78d6", "#0d366b"], as_cmap=True)
DIV = sns.blend_palette(["#2a78d6", "#f0efec", "#e34948"], as_cmap=True)
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titlecolor": INK, "axes.titleweight": "bold", "axes.titlesize": 12,
    "axes.titlelocation": "left", "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.prop_cycle": plt.cycler(color=SERIES),
    "lines.linewidth": 2, "figure.dpi": 110, "font.size": 10, "legend.frameon": False,
})
usd = mtick.StrMethodFormatter("${x:,.0f}")
import matplotlib.dates as mdates
def fmt_dates(ax):
    loc = mdates.AutoDateLocator(minticks=3, maxticks=8); ax.xaxis.set_major_locator(loc); ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
def savefig(fig, name): fig.savefig(FIG / f"{name}.png", bbox_inches="tight")
import json, joblib, warnings
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings("ignore")
FINAL = ROOT / "data/final"; MODELS = ROOT / "models/final"; METRICS = ROOT / "reports/metrics"; REF = ROOT / "data/reference"
spec = json.loads((FINAL / "feature_spec.json").read_text()); SEED = 42
acc = lambda y, p: 100 - np.abs(np.asarray(y) - np.asarray(p)).sum() / np.abs(np.asarray(y)).sum() * 100
def score(y, p): return {"R2": r2_score(y, p), "accuracy_%": acc(y, p), "MAE": mean_absolute_error(y, p)}
def gbr(**kw): return HistGradientBoostingRegressor(random_state=SEED, early_stopping=False, **kw)

p_train = pd.read_csv(FINAL / "product_train.csv"); p_test = pd.read_csv(FINAL / "product_test.csv")
y_train = pd.read_csv(FINAL / "yearly_train.csv");  y_test = pd.read_csv(FINAL / "yearly_test.csv")
for d in (p_train, p_test):
    d["log_listing_age"] = np.log1p(d.age_months_at_obs_end); d["log_reviews_per_month"] = d.log_rating_count - d.log_listing_age
LF = spec["product"]["launch_features"] + ["log_rating_count", "log_listing_age", "log_reviews_per_month"]
YF = spec["yearly"]["features"]

## 1. Category market size (new reference input)
`demand_share_pct_12m` is a product's share of its category's demand, so the model needs the **size of the category market**.
We compute each category's total monthly demand from the full monthly history (`data/processed/monthly_panel.csv`) and save it to
`data/reference/category_monthly_demand.csv`. In the app, a new product uses its category's most recent 12-month average.

In [2]:
panel = pd.read_csv(ROOT / "data/processed/monthly_panel.csv", usecols=["product_id", "month", "age_months", "demand_units"], parse_dates=["month"])
cats = pd.concat([p_train, p_test])[["product_id", "us_category"]]
panel = panel.merge(cats, on="product_id", how="inner")
market = panel.groupby(["us_category", "month"]).demand_units.sum().rename("category_demand_units").reset_index()
market.to_csv(REF / "category_monthly_demand.csv", index=False)
first_year = panel[panel.age_months < 12].merge(market, on=["us_category", "month"])
size = first_year.groupby("product_id").category_demand_units.mean().rename("cat_market_size_12m")
for d in (p_train, p_test):
    d["cat_market_size_12m"] = d.product_id.map(size); d["log_cat_market_size"] = np.log1p(d.cat_market_size_12m)
print("market size available:", p_train.cat_market_size_12m.notna().mean().round(3), "of train products")
market.groupby("us_category").category_demand_units.mean().sort_values(ascending=False).round(0).head(8)

market size available: 1.0 of train products


us_category
Cell Phones & Accessories             628195.0
Electronics - Audio                   520737.0
Computer Accessories & Peripherals    437258.0
Computers & Accessories               361266.0
Kitchen & Home Appliances             358788.0
Electronics - TV & Video              284778.0
Electronics - Accessories             126433.0
Home & Kitchen                         55992.0
Name: category_demand_units, dtype: float64

## 2. Break-even months (capped at 36): choose the setup with 5-fold CV (train only)

In [3]:
TE = ["brand_te_log_revenue", "brand_te_log_units", "brand_te_margin", "leaf_te_log_revenue", "leaf_te_log_units"]
SETUPS = {"v2 (notebook 06)": dict(learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=40, l2_regularization=1.0, max_iter=500),
          "small leaves, lr 0.05": dict(learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=1.0, max_iter=400),
          "small leaves, lr 0.05, 700 trees": dict(learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=1.0, max_iter=700)}
BE_CAP = 36   # months; longer paybacks are reported as "36+ months"
be = p_train.dropna(subset=["break_even_months"]).reset_index(drop=True)
print(f"Products with break-even > {BE_CAP} months: {(be.break_even_months > BE_CAP).mean():.1%}")
for d in (p_train, p_test): d["break_even_capped"] = d.break_even_months.clip(upper=BE_CAP)
be["break_even_capped"] = be.break_even_months.clip(upper=BE_CAP)
cv = []
for name, prm in SETUPS.items():
    s = []
    for tr_i, va_i in StratifiedKFold(5, shuffle=True, random_state=SEED).split(be, be.category_group):
        m = gbr(**prm).fit(be.loc[tr_i, LF], np.log1p(be.loc[tr_i, "break_even_capped"]))
        s.append(score(be.loc[va_i, "break_even_capped"], np.minimum(np.expm1(m.predict(be.loc[va_i, LF])), BE_CAP)))
    cv.append({"setup": name} | pd.DataFrame(s).mean().to_dict())
cv = pd.DataFrame(cv); BEST_BE = cv.sort_values("accuracy_%", ascending=False).setup.iloc[0]
print("chosen:", BEST_BE); cv.round(3)

Products with break-even > 36 months: 5.9%


chosen: small leaves, lr 0.05, 700 trees


,setup,R2,accuracy_%,MAE
0,v2 (notebook 06),0.785,72.969,2.221
1,"small leaves, lr 0.05",0.794,74.012,2.144
2,"small leaves, lr 0.05, 700 trees",0.795,74.107,2.135


## 3. Validation of the two composites (train only)

In [4]:
# Demand share: composite = predicted avg monthly demand / category market size × 100
SF = LF + ["log_cat_market_size"]
ds = p_train.dropna(subset=["demand_share_pct_12m", "cat_market_size_12m"]).reset_index(drop=True)
DS_SETUPS = {"lr 0.05, leaf 20, 400": dict(learning_rate=0.05, min_samples_leaf=20, max_iter=400),
             "lr 0.05, leaf 10, 600": dict(learning_rate=0.05, min_samples_leaf=10, max_iter=600),
             "lr 0.03, leaf 15, 900": dict(learning_rate=0.03, min_samples_leaf=15, max_iter=900, l2_regularization=1.0)}
ds_cv = []
for name, prm in DS_SETUPS.items():
    s = []
    for tr_i, va_i in StratifiedKFold(5, shuffle=True, random_state=SEED).split(ds, ds.category_group):
        m = gbr(**prm).fit(ds.loc[tr_i, SF], np.log1p(ds.loc[tr_i, "avg_monthly_demand_12m"]))
        p = np.expm1(m.predict(ds.loc[va_i, SF])) / ds.loc[va_i, "cat_market_size_12m"] * 100
        s.append(score(ds.loc[va_i, "demand_share_pct_12m"], p))
    ds_cv.append({"setup": name} | pd.DataFrame(s).mean().to_dict())
ds_cv = pd.DataFrame(ds_cv); BEST_DS = DS_SETUPS[ds_cv.sort_values("accuracy_%", ascending=False).setup.iloc[0]]
print("demand share composite, 5-fold CV:"); print(ds_cv.round(3).to_string())

# Yearly profit: composite = (last year's revenue + predicted change) × predicted margin; fit ≤ 2023, validate 2024
def fit_yearly_profit(df):
    has_prev = df.gross_revenue_prev_year.fillna(0) > 0
    growth = gbr(learning_rate=0.05, max_iter=600, min_samples_leaf=20).fit(
        df[has_prev][YF], np.log(df[has_prev].gross_revenue.clip(lower=1) / df[has_prev].gross_revenue_prev_year))
    level = gbr(learning_rate=0.1, max_iter=600).fit(df[YF], np.log1p(df.gross_revenue))
    margin = gbr(learning_rate=0.1, max_iter=300).fit(df[YF], df.profit_margin_pct)
    return {"growth": growth, "level": level, "margin": margin}
def predict_yearly_profit(m, X):
    has_prev = (X.gross_revenue_prev_year.fillna(0) > 0).values
    revenue = np.where(has_prev, X.gross_revenue_prev_year.fillna(0) * np.exp(m["growth"].predict(X[YF])), np.expm1(m["level"].predict(X[YF])))
    return revenue * m["margin"].predict(X[YF]) / 100
tr, va = y_train[y_train.year <= 2023], y_train[y_train.year == 2024]
p = predict_yearly_profit(fit_yearly_profit(tr), va)
est = (va.months_active == 12) & (va.months_active_prev_year == 12)
print("yearly profit composite, validation 2024 (all rows):        ", {k: round(v, 3) for k, v in score(va.net_profit, p).items()})
print("yearly profit composite, validation 2024 (full prior year): ", {k: round(v, 3) for k, v in score(va.net_profit[est], p[est.values]).items()})

demand share composite, 5-fold CV:
                   setup     R2  accuracy_%    MAE
0  lr 0.05, leaf 20, 400  0.815      69.989  0.896
1  lr 0.05, leaf 10, 600  0.826      70.725  0.868
2  lr 0.03, leaf 15, 900  0.839      71.469  0.847


yearly profit composite, validation 2024 (all rows):         {'R2': 0.93, 'accuracy_%': np.float64(73.087), 'MAE': 292278.875}
yearly profit composite, validation 2024 (full prior year):  {'R2': 0.928, 'accuracy_%': np.float64(82.097), 'MAE': 68286.634}


## 4. Final test (once) → retrain on all data → overwrite these three models in `models/final/`

In [5]:
res = []
# --- break-even
prm = SETUPS[BEST_BE]
tr_be, te_be = p_train.dropna(subset=["break_even_months"]), p_test.dropna(subset=["break_even_months"])
m = gbr(**prm).fit(tr_be[LF], np.log1p(tr_be.break_even_capped)); p = np.minimum(np.expm1(m.predict(te_be[LF])), BE_CAP)
res.append({"level": "product", "target": "break_even_months"} | score(te_be.break_even_capped, p))
# --- demand share (composite)
tr_ds = p_train.dropna(subset=["demand_share_pct_12m", "cat_market_size_12m"]); te_ds = p_test.dropna(subset=["demand_share_pct_12m", "cat_market_size_12m"])
m = gbr(**BEST_DS).fit(tr_ds[SF], np.log1p(tr_ds.avg_monthly_demand_12m))
p = np.expm1(m.predict(te_ds[SF])) / te_ds.cat_market_size_12m * 100
res.append({"level": "product", "target": "demand_share_pct_12m"} | score(te_ds.demand_share_pct_12m, p))
# --- yearly profit (composite)
p = predict_yearly_profit(fit_yearly_profit(y_train), y_test)
est = ((y_test.months_active == 12) & (y_test.months_active_prev_year == 12)).values
res.append({"level": "yearly", "target": "net_profit"} | score(y_test.net_profit, p))
res.append({"level": "yearly", "target": "net_profit (products with a full prior year)"} | score(y_test.net_profit[est], p[est]))
v3 = pd.DataFrame(res)
v2 = pd.read_csv(METRICS / "model_metrics_v2.csv")
v2_acc = {}
for _, r in v2.iterrows():
    d = {"product": p_test, "yearly": y_test}.get(r.level)
    if d is not None and r.target in d: v2_acc[(r.level, r.target)] = 100 - r.test_MAE / d[r.target].abs().mean() * 100
v3["v2_accuracy_%"] = [v2_acc.get((r.level, r.target.split(" (")[0])) for _, r in v3.iterrows()]
v3["v2_R2"] = [v2.set_index(["level", "target"]).test_R2.get((r.level, r.target.split(" (")[0])) for _, r in v3.iterrows()]
v3.to_csv(METRICS / "model_metrics_v3_targeted.csv", index=False)
v3.round(3)

,level,target,R2,accuracy_%,MAE,v2_accuracy_%,v2_R2
0,product,break_even_months,0.806,75.081,2.173,57.165,0.490
1,product,demand_share_pct_12m,0.904,72.573,0.731,53.668,0.777
2,yearly,net_profit,0.948,78.357,236389.865,59.644,0.778
3,yearly,net_profit (products with a full prior year),0.990,89.277,72866.490,59.644,0.778


In [6]:
# Retrain on ALL data (train + test) and overwrite the three bundles
p_all, y_all = pd.concat([p_train, p_test], ignore_index=True), pd.concat([y_train, y_test], ignore_index=True)
d = p_all.dropna(subset=["break_even_months"])
d = d.assign(break_even_capped=d.break_even_months.clip(upper=BE_CAP))
joblib.dump({"model": gbr(**prm).fit(d[LF], np.log1p(d.break_even_capped)), "features": LF, "target": "break_even_months", "level": "product",
             "transform": "log1p", "cap_months": BE_CAP, "base_col": None, "classifier": False, "composite": None, "algorithm": "HistGradientBoostingRegressor",
             "inputs_note": "needs log_rating_count, log_listing_age, log_reviews_per_month; output capped at 36 = '36+ months'"},
            MODELS / "product" / "break_even_months.joblib")
d = p_all.dropna(subset=["demand_share_pct_12m", "cat_market_size_12m"])
joblib.dump({"kind": "composite_share", "demand_model": gbr(**BEST_DS).fit(d[SF], np.log1p(d.avg_monthly_demand_12m)),
             "features": SF, "target": "demand_share_pct_12m", "level": "product",
             "formula": "expm1(demand_model) / cat_market_size_12m * 100", "market_reference": "data/reference/category_monthly_demand.csv"},
            MODELS / "product" / "demand_share_pct_12m.joblib")
joblib.dump({"kind": "composite_profit", "models": fit_yearly_profit(y_all), "features": YF, "target": "net_profit", "level": "yearly",
             "formula": "revenue × margin/100; revenue = prev-year revenue × exp(growth) if a previous year exists, else expm1(level)",
             "use_note": "most reliable for products with a full previous year; for first-year products use the product-launch models"},
            MODELS / "yearly" / "net_profit.joblib")
reg = pd.read_csv(MODELS / "model_registry.csv")
for _, r in v3[~v3.target.str.contains("full prior")].iterrows():
    i = (reg.level == r.level) & (reg.target == r.target)
    reg.loc[i, ["test_R2", "test_MAE"]] = [r.R2, r.MAE]
    reg.loc[i, "algorithm"] = {"break_even_months": "HistGradientBoostingRegressor", "demand_share_pct_12m": "composite: demand / market size",
                               "net_profit": "composite: (prev revenue × growth) × margin"}[r.target]
reg.to_csv(MODELS / "model_registry.csv", index=False)
print("Overwrote 3 models in models/final and updated model_registry.csv")

Overwrote 3 models in models/final and updated model_registry.csv


## 5. Why the two demand-growth models can't reach 70% / R² 0.65
The simulated history (notebook 03) adds realistic month-to-month randomness: lognormal noise with σ = 0.18 and AR(1) φ = 0.5.
That randomness alone sets a floor on how predictable *growth rates* can be:

In [7]:
sigma, phi = 0.18, 0.5
month_sd = sigma / np.sqrt(1 - phi**2)                          # sd of one month's log-noise
year_sd = np.sqrt(month_sd**2 / 12 * (1 + phi) / (1 - phi))       # sd of a yearly average of that noise
yoy_noise_pp = np.sqrt(2) * year_sd * 100                         # two independent years -> growth noise, in % points
actual_sd = y_test.demand_growth_yoy_pct.std()
print(f"Pure-noise spread of YoY growth: ~{yoy_noise_pp:.1f} pp   |   actual spread in test data: {actual_sd:.1f} pp")
print(f"=> about {min(1, (yoy_noise_pp / actual_sd) ** 2):.0%} of YoY growth variance is unpredictable noise; the best possible R² is roughly {max(0, 1 - (yoy_noise_pp / actual_sd) ** 2):.2f}")

Pure-noise spread of YoY growth: ~14.7 pp   |   actual spread in test data: 15.1 pp
=> about 95% of YoY growth variance is unpredictable noise; the best possible R² is roughly 0.05


**Conclusion:** most of the growth signal is random by construction, so no model can honestly reach R² 0.65 or 70% accuracy on these two targets.
The demand **level** models (monthly 82%, yearly 76%) are what the app should rely on. Growth % should be shown as an indicative trend.
The only way to "hit" the target would be to reduce the noise in the simulated data, which would make the data less realistic. We don't do that.